# DEEP NEURAL NETWORKS - ASSIGNMENT 3: RNN vs TRANSFORMER FOR TIME SERIES
## Recurrent Neural Networks vs Transformers for Time Series Prediction

**BITS ID:** 2025AF05094

**Name:** NAAZ VERMA

**Email:** 2025af05094@wilp.bits-pilani.ac.in

**Date:** 2026-04-19


In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import json
import os
import math

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Input, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")


---
## PART 1: DATASET LOADING AND EXPLORATION

**Dataset:** Stock Market (Yahoo Finance - AAPL)

Using `yfinance` to download Apple stock price data. We predict the **Close** price.


In [ ]:
# 1.1 Install yfinance and Download Stock Data
!pip install yfinance -q

import yfinance as yf

# Download Apple stock data (5 years of daily data)
ticker = "AAPL"
df = yf.download(ticker, start="2019-01-01", end="2024-12-31")
df = df[['Close']].copy()
df.dropna(inplace=True)
df.reset_index(inplace=True)

print(f"Downloaded {len(df)} rows of {ticker} stock data")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(df.head())
print(df.describe())


In [ ]:
# 1.2 Dataset Metadata
dataset_name = "Apple (AAPL) Stock Price"
dataset_source = "Yahoo Finance (yfinance)"
n_samples = len(df)
n_features = 1  # Univariate: Close price only
sequence_length = 30  # Lookback window: 30 days
prediction_horizon = 1  # Predict 1 day ahead
problem_type = "time_series_forecasting"

primary_metric = "MAE"
metric_justification = (
    "MAE is chosen because it measures average prediction error in the same units as "
    "the stock price (dollars), making it directly interpretable. Unlike RMSE, MAE is "
    "less sensitive to outliers which are common in stock data."
)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)
print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Total Samples: {n_samples}")
print(f"Number of Features: {n_features}")
print(f"Sequence Length: {sequence_length}")
print(f"Prediction Horizon: {prediction_horizon}")
print(f"Primary Metric: {primary_metric}")
print(f"Metric Justification: {metric_justification}")


In [ ]:
# 1.3 Time Series Exploration
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Full time series
axes[0, 0].plot(df['Date'], df['Close'].values, color='steelblue')
axes[0, 0].set_title(f'{ticker} Closing Price')
axes[0, 0].set_xlabel('Date'); axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(True)

# Distribution
axes[0, 1].hist(df['Close'].values.flatten(), bins=50, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Price Distribution')
axes[0, 1].set_xlabel('Price ($)'); axes[0, 1].set_ylabel('Frequency')

# Daily returns
df['Returns'] = df['Close'].pct_change()
axes[1, 0].plot(df['Date'], df['Returns'].values, color='coral', alpha=0.7)
axes[1, 0].set_title('Daily Returns')
axes[1, 0].set_xlabel('Date'); axes[1, 0].set_ylabel('Return')
axes[1, 0].grid(True)

# Rolling statistics
rolling_mean = df['Close'].rolling(window=50).mean().values.flatten()
rolling_std = df['Close'].rolling(window=50).std().values.flatten()
close_vals = df['Close'].values.flatten()
axes[1, 1].plot(df['Date'], close_vals, label='Close Price', alpha=0.7)
axes[1, 1].plot(df['Date'], rolling_mean, label='50-day MA', color='red')
axes[1, 1].fill_between(df['Date'], rolling_mean - 2*rolling_std, rolling_mean + 2*rolling_std,
                         alpha=0.1, color='red')
axes[1, 1].set_title('Price with 50-day Moving Average')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nPrice Statistics:")
print(f"  Min: ${df['Close'].min().item():.2f}")
print(f"  Max: ${df['Close'].max().item():.2f}")
print(f"  Mean: ${df['Close'].mean().item():.2f}")
print(f"  Std: ${df['Close'].std().item():.2f}")

In [ ]:
# 1.4 Data Preprocessing
# Use only Close price
close_prices = df['Close'].values.reshape(-1, 1)

# Scale data to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_prices)

# Create sequences using sliding window
def create_sequences(data, seq_length, pred_horizon=1):
    X, y = [], []
    for i in range(len(data) - seq_length - pred_horizon + 1):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length:i+seq_length+pred_horizon])
    return np.array(X), np.array(y).reshape(-1, pred_horizon)

X, y = create_sequences(scaled_data, sequence_length, prediction_horizon)
print(f"Total sequences: {len(X)}")
print(f"X shape: {X.shape}")  # (samples, seq_length, features)
print(f"y shape: {y.shape}")  # (samples, pred_horizon)

# Temporal split: 90/10 (NO SHUFFLING - time series must be in order)
split_idx = int(len(X) * 0.9)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_test_ratio = "90/10"
train_samples = len(X_train)
test_samples = len(X_test)

print(f"\nTrain/Test Split: {train_test_ratio}")
print(f"Training Samples: {train_samples}")
print(f"Test Samples: {test_samples}")
print("IMPORTANT: Temporal split used (NO shuffling)")


In [ ]:
# 1.5 Visualize Train/Test Split
fig, ax = plt.subplots(figsize=(14, 5))
train_dates = df['Date'].iloc[sequence_length:sequence_length+split_idx]
test_dates = df['Date'].iloc[sequence_length+split_idx:sequence_length+len(X)]

# Inverse transform for plotting
y_train_actual = scaler.inverse_transform(y_train)
y_test_actual = scaler.inverse_transform(y_test)

ax.plot(train_dates, y_train_actual, label='Train', color='steelblue')
ax.plot(test_dates, y_test_actual, label='Test', color='coral')
ax.axvline(x=test_dates.iloc[0], color='black', linestyle='--', label='Split Point')
ax.set_title('Train/Test Split (Temporal)')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()


---
## PART 2: LSTM IMPLEMENTATION (5 Marks)

Using 2 stacked LSTM layers with Dropout for regularization.


In [ ]:
# 2.1 LSTM Architecture
rnn_model_type = "LSTM"

def build_rnn_model(input_shape, hidden_units=64, n_layers=2, output_size=1):
    model = Sequential()
    # First LSTM layer (return sequences for stacking)
    model.add(LSTM(hidden_units, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    # Second LSTM layer
    model.add(LSTM(hidden_units, return_sequences=False))
    model.add(Dropout(0.2))
    # Output layer
    model.add(Dense(output_size))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

rnn_model = build_rnn_model(
    input_shape=(sequence_length, n_features),
    hidden_units=64, n_layers=2, output_size=prediction_horizon
)
rnn_model.summary()


In [ ]:
# 2.2 Train LSTM Model
print("=" * 70)
print("LSTM MODEL TRAINING")
print("=" * 70)

rnn_epochs = 50
rnn_batch_size = 32
rnn_start_time = time.time()

history_rnn = rnn_model.fit(
    X_train, y_train,
    epochs=rnn_epochs,
    batch_size=rnn_batch_size,
    validation_split=0.1,
    verbose=1
)

rnn_training_time = time.time() - rnn_start_time
rnn_initial_loss = history_rnn.history['loss'][0]
rnn_final_loss = history_rnn.history['loss'][-1]

print(f"\nTraining completed in {rnn_training_time:.2f} seconds")
print(f"Initial Loss: {rnn_initial_loss:.6f}")
print(f"Final Loss: {rnn_final_loss:.6f}")


In [ ]:
# 2.3 Evaluate LSTM Model
y_pred_rnn_scaled = rnn_model.predict(X_test)

# Inverse transform predictions and actuals back to original scale
y_pred_rnn = scaler.inverse_transform(y_pred_rnn_scaled)
y_test_orig = scaler.inverse_transform(y_test)

def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

rnn_mae = mean_absolute_error(y_test_orig, y_pred_rnn)
rnn_rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_rnn))
rnn_mape = calculate_mape(y_test_orig, y_pred_rnn)
rnn_r2 = r2_score(y_test_orig, y_pred_rnn)

print("=" * 70)
print("LSTM MODEL EVALUATION")
print("=" * 70)
print(f"  MAE:      ${rnn_mae:.4f}")
print(f"  RMSE:     ${rnn_rmse:.4f}")
print(f"  MAPE:     {rnn_mape:.4f}%")
print(f"  R2 Score: {rnn_r2:.4f}")


In [ ]:
# 2.4 Visualize LSTM Results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss
axes[0].plot(history_rnn.history['loss'], label='Train Loss')
axes[0].plot(history_rnn.history['val_loss'], label='Val Loss')
axes[0].set_title('LSTM - Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].legend(); axes[0].grid(True)

# Actual vs Predicted
axes[1].plot(y_test_orig, label='Actual', color='steelblue')
axes[1].plot(y_pred_rnn, label='Predicted', color='coral', alpha=0.8)
axes[1].set_title('LSTM - Actual vs Predicted')
axes[1].set_xlabel('Time Step'); axes[1].set_ylabel('Price ($)')
axes[1].legend(); axes[1].grid(True)

# Residuals
residuals_rnn = y_test_orig.flatten() - y_pred_rnn.flatten()
axes[2].plot(residuals_rnn, color='steelblue', alpha=0.7)
axes[2].axhline(y=0, color='red', linestyle='--')
axes[2].set_title('LSTM - Residuals')
axes[2].set_xlabel('Time Step'); axes[2].set_ylabel('Error ($)')
axes[2].grid(True)

plt.tight_layout()
plt.show()


---
## PART 3: TRANSFORMER IMPLEMENTATION (5 Marks)

Using Transformer encoder with:
- **Sinusoidal Positional Encoding (MANDATORY)**
- `MultiHeadAttention` from Keras
- Feed-forward layers


In [ ]:
# 3.1 Positional Encoding Implementation
def positional_encoding(seq_length, d_model):
    """Generate sinusoidal positional encodings."""
    positions = np.arange(seq_length)[:, np.newaxis]  # (seq_length, 1)
    dims = np.arange(d_model)[np.newaxis, :]  # (1, d_model)

    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    # Apply sin to even indices, cos to odd indices
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])

    return angles.astype(np.float32)

# Visualize positional encoding
pe = positional_encoding(sequence_length, 64)
plt.figure(figsize=(12, 4))
plt.imshow(pe, cmap='viridis', aspect='auto')
plt.colorbar(label='Value')
plt.title('Sinusoidal Positional Encoding')
plt.xlabel('Encoding Dimension'); plt.ylabel('Position')
plt.tight_layout()
plt.show()
print(f"Positional encoding shape: {pe.shape}")


In [ ]:
# 3.2 Transformer Encoder Model
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, d_ff, dropout_rate=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)
        self.ffn = tf.keras.Sequential([
            Dense(d_ff, activation='relu'),
            Dense(d_model)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, x, training=False):
        # Multi-head self-attention
        attn_output = self.mha(x, x, x)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        # Feed-forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

def build_transformer_model(seq_length, n_features, d_model=64, n_heads=4,
                             n_layers=2, d_ff=128, output_size=1, dropout_rate=0.1):
    inputs = Input(shape=(seq_length, n_features))

    # Project input to d_model dimensions
    x = Dense(d_model)(inputs)

    # Add positional encoding (MANDATORY)
    pos_enc = positional_encoding(seq_length, d_model)
    pos_enc_tensor = tf.constant(pos_enc, dtype=tf.float32)
    x = x + pos_enc_tensor

    # Transformer encoder blocks
    for _ in range(n_layers):
        x = TransformerBlock(d_model, n_heads, d_ff, dropout_rate)(x)

    # Global average pooling over time dimension
    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(output_size)(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Model hyperparameters
t_d_model = 64
t_n_heads = 4
t_n_layers = 2
t_d_ff = 128

transformer_model = build_transformer_model(
    sequence_length, n_features, d_model=t_d_model, n_heads=t_n_heads,
    n_layers=t_n_layers, d_ff=t_d_ff, output_size=prediction_horizon
)
transformer_model.summary()


In [ ]:
# 3.3 Train Transformer Model
print("=" * 70)
print("TRANSFORMER MODEL TRAINING")
print("=" * 70)

transformer_epochs = 50
transformer_batch_size = 32
transformer_start_time = time.time()

history_transformer = transformer_model.fit(
    X_train, y_train,
    epochs=transformer_epochs,
    batch_size=transformer_batch_size,
    validation_split=0.1,
    verbose=1
)

transformer_training_time = time.time() - transformer_start_time
transformer_initial_loss = history_transformer.history['loss'][0]
transformer_final_loss = history_transformer.history['loss'][-1]

print(f"\nTraining completed in {transformer_training_time:.2f} seconds")
print(f"Initial Loss: {transformer_initial_loss:.6f}")
print(f"Final Loss: {transformer_final_loss:.6f}")


In [ ]:
# 3.4 Evaluate Transformer Model
y_pred_transformer_scaled = transformer_model.predict(X_test)
y_pred_transformer = scaler.inverse_transform(y_pred_transformer_scaled)

transformer_mae = mean_absolute_error(y_test_orig, y_pred_transformer)
transformer_rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_transformer))
transformer_mape = calculate_mape(y_test_orig, y_pred_transformer)
transformer_r2 = r2_score(y_test_orig, y_pred_transformer)

print("=" * 70)
print("TRANSFORMER MODEL EVALUATION")
print("=" * 70)
print(f"  MAE:      ${transformer_mae:.4f}")
print(f"  RMSE:     ${transformer_rmse:.4f}")
print(f"  MAPE:     {transformer_mape:.4f}%")
print(f"  R2 Score: {transformer_r2:.4f}")


In [ ]:
# 3.5 Visualize Transformer Results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_transformer.history['loss'], label='Train Loss')
axes[0].plot(history_transformer.history['val_loss'], label='Val Loss')
axes[0].set_title('Transformer - Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(y_test_orig, label='Actual', color='steelblue')
axes[1].plot(y_pred_transformer, label='Predicted', color='coral', alpha=0.8)
axes[1].set_title('Transformer - Actual vs Predicted')
axes[1].set_xlabel('Time Step'); axes[1].set_ylabel('Price ($)')
axes[1].legend(); axes[1].grid(True)

residuals_transformer = y_test_orig.flatten() - y_pred_transformer.flatten()
axes[2].plot(residuals_transformer, color='coral', alpha=0.7)
axes[2].axhline(y=0, color='red', linestyle='--')
axes[2].set_title('Transformer - Residuals')
axes[2].set_xlabel('Time Step'); axes[2].set_ylabel('Error ($)')
axes[2].grid(True)

plt.tight_layout()
plt.show()


---
## PART 4: MODEL COMPARISON AND VISUALIZATION


In [ ]:
# 4.1 Metrics Comparison
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

rnn_total_params = rnn_model.count_params()
transformer_total_params = transformer_model.count_params()

comparison_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'R2 Score', 'Training Time (s)', 'Parameters'],
    'LSTM': [rnn_mae, rnn_rmse, rnn_mape, rnn_r2, rnn_training_time, rnn_total_params],
    'Transformer': [transformer_mae, transformer_rmse, transformer_mape, transformer_r2,
                    transformer_training_time, transformer_total_params]
})
print(comparison_df.to_string(index=False))


In [ ]:
# 4.2 Visual Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar plot - metrics
metrics_names = ['MAE', 'RMSE', 'MAPE (%)']
rnn_vals = [rnn_mae, rnn_rmse, rnn_mape]
tr_vals = [transformer_mae, transformer_rmse, transformer_mape]
x = np.arange(len(metrics_names))
width = 0.35
axes[0].bar(x - width/2, rnn_vals, width, label='LSTM', color='steelblue')
axes[0].bar(x + width/2, tr_vals, width, label='Transformer', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_names)
axes[0].set_title('Error Metrics (lower is better)')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Both predictions vs actual
axes[1].plot(y_test_orig, label='Actual', color='black', linewidth=1.5)
axes[1].plot(y_pred_rnn, label='LSTM', color='steelblue', alpha=0.8)
axes[1].plot(y_pred_transformer, label='Transformer', color='coral', alpha=0.8)
axes[1].set_title('Predictions Comparison')
axes[1].set_xlabel('Time Step'); axes[1].set_ylabel('Price ($)')
axes[1].legend(); axes[1].grid(True)

# Training loss comparison
axes[2].plot(history_rnn.history['loss'], label='LSTM', color='steelblue')
axes[2].plot(history_transformer.history['loss'], label='Transformer', color='coral')
axes[2].set_title('Training Loss Comparison')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE Loss')
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()


---
## PART 5: ANALYSIS (2 Marks)


In [ ]:
analysis_text = """
The Transformer dramatically outperformed the LSTM across all metrics: MAE ($3.11 vs
$10.65), RMSE ($3.98 vs $11.60), MAPE (1.41% vs 4.70%), and R2 (0.91 vs 0.24). The
LSTM's low R2 of 0.24 indicates it captured only 24% of price variance, while the
Transformer explained 91% — a substantial difference.

The Transformer's self-attention mechanism computes pairwise relationships across all
30 time steps simultaneously, enabling it to identify which past days are most predictive.
This global view proved critical for capturing the test period's price dynamics. The LSTM
processes data sequentially through gates, struggling with longer dependencies even within
a 30-step window.

Sinusoidal positional encoding was essential for the Transformer since attention is
permutation-invariant. Without it, the model would treat all time positions identically,
destroying temporal order. The encoding injects unique position signals using sin/cos
functions at varying frequencies.

Both models achieved excellent convergence: LSTM reduced loss by 96.6% (0.0254 to
0.0009) and the Transformer by 99.8% (0.5111 to 0.0009). The Transformer's higher
initial loss reflects its random initialization before learning attention patterns.

Computationally, the LSTM trained in 27.5s with 49,985 parameters versus 35.5s and
67,137 parameters for the Transformer. The modest 8-second difference is negligible
given the Transformer's vastly superior accuracy, making it the clear winner for this
stock prediction task.
"""

print("=" * 70)
print("ANALYSIS")
print("=" * 70)
print(analysis_text)
word_count = len(analysis_text.split())
print(f"Analysis word count: {word_count} words")
if word_count > 200:
    print("  Note: Slightly exceeds 200-word guideline (no marks deduction per instructions)")
else:
    print("  Within word count guideline")

---
## PART 6: ASSIGNMENT RESULTS SUMMARY (Auto-Grading)


In [ ]:
def get_assignment_results():
    framework_used = "keras"
    results = {
        'dataset_name': dataset_name, 'dataset_source': dataset_source,
        'n_samples': n_samples, 'n_features': n_features,
        'sequence_length': sequence_length, 'prediction_horizon': prediction_horizon,
        'problem_type': problem_type, 'primary_metric': primary_metric,
        'metric_justification': metric_justification,
        'train_samples': train_samples, 'test_samples': test_samples,
        'train_test_ratio': train_test_ratio,
        'rnn_model': {
            'framework': framework_used, 'model_type': rnn_model_type,
            'architecture': {
                'n_layers': 2, 'hidden_units': 64, 'total_parameters': rnn_total_params
            },
            'training_config': {
                'learning_rate': 0.001, 'n_epochs': rnn_epochs,
                'batch_size': rnn_batch_size, 'optimizer': 'Adam',
                'loss_function': 'MSE'
            },
            'initial_loss': rnn_initial_loss, 'final_loss': rnn_final_loss,
            'training_time_seconds': rnn_training_time,
            'mae': rnn_mae, 'rmse': rnn_rmse,
            'mape': rnn_mape, 'r2_score': rnn_r2
        },
        'transformer_model': {
            'framework': framework_used,
            'architecture': {
                'n_layers': t_n_layers, 'n_heads': t_n_heads,
                'd_model': t_d_model, 'd_ff': t_d_ff,
                'has_positional_encoding': True, 'has_attention': True,
                'total_parameters': transformer_total_params
            },
            'training_config': {
                'learning_rate': 0.001, 'n_epochs': transformer_epochs,
                'batch_size': transformer_batch_size, 'optimizer': 'Adam',
                'loss_function': 'MSE'
            },
            'initial_loss': transformer_initial_loss, 'final_loss': transformer_final_loss,
            'training_time_seconds': transformer_training_time,
            'mae': transformer_mae, 'rmse': transformer_rmse,
            'mape': transformer_mape, 'r2_score': transformer_r2
        },
        'analysis': analysis_text,
        'analysis_word_count': len(analysis_text.split()),
        'rnn_loss_decreased': rnn_final_loss < rnn_initial_loss,
        'transformer_loss_decreased': transformer_final_loss < transformer_initial_loss,
    }
    return results

try:
    assignment_results = get_assignment_results()
    print("=" * 70)
    print("ASSIGNMENT RESULTS SUMMARY")
    print("=" * 70)
    print(json.dumps(assignment_results, indent=2))
except Exception as e:
    print(f"ERROR generating results: {str(e)}")
    print("Please ensure all variables are properly defined")


In [ ]:
# ENVIRONMENT VERIFICATION
import platform
import sys
from datetime import datetime

print("ENVIRONMENT INFORMATION")
print(f"  Python: {sys.version}")
print(f"  TensorFlow: {tf.__version__}")
print(f"  NumPy: {np.__version__}")
print(f"  Platform: {platform.platform()}")
print(f"  Timestamp: {datetime.now().isoformat()}")
print()
print("REQUIRED: Add screenshot of your Google Colab/BITS Virtual Lab")
print("showing your account details in the cell below this one.")


### Environment Screenshot

*Paste screenshot of Google Colab showing your account details here.*
